# video2rig · Colab 核心训练
本地 Blender 生成姿态、图片和可见性标签，Colab 只训练 joint reader。无需在 Colab 安装 Blender。

1. 运行时选择 **GPU**（T4 可以使用；支持原生 BF16 的卡更快）。
2. 本地执行 `python -m video2rig.reader.prepare my_troll --blend model.blend --skip-train`，再执行 `python train_colab.py pack my_troll --output work/training.zip`。
3. 运行全部单元格：默认从 GitHub 的 colab-training 分支获取代码，按提示上传 `training.zip`；训练后自动下载权重。

如需使用本地修改的源码，可把 source_mode 改成 upload，并上传 `python train_colab.py source` 生成的源码包。只上传可信来源的项目和权重。

In [ ]:
#@title 训练配置
source_mode = "github" #@param ["upload", "github"]
repo_url = "https://github.com/GrinZero/video2rig.git" #@param {type:"string"}
repo_ref = "colab-training" #@param {type:"string"}
# 留空时上传 training.zip；也可以填写 /content/drive/MyDrive/.../training.zip
training_zip = "" #@param {type:"string"}
use_drive = False #@param {type:"boolean"}
steps = 9000 #@param {type:"integer"}
minutes = 8.0 #@param {type:"number"}
batch = 4 #@param {type:"integer"}
seed = 0 #@param {type:"integer"}
# 若启用 Drive，将结果直接保存到 Drive。steps=0 才使用 minutes。


In [ ]:
#@title 准备代码与依赖（保留 Colab 自带 CUDA PyTorch）
import sys, subprocess, zipfile, tempfile
from pathlib import Path
from google.colab import files
import torch

if not torch.cuda.is_available():
    raise RuntimeError("请选择：运行时 → 更改运行时类型 → GPU")
print("GPU:", torch.cuda.get_device_name(0), "原生 BF16:", torch.cuda.is_bf16_supported())
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')

def run(args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)

def upload_one():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("请一次只选择一个 ZIP 文件")
    return Path(next(iter(uploaded))).resolve()

session = Path(tempfile.mkdtemp(prefix='video2rig-', dir='/content'))
if source_mode == 'upload':
    print("请上传 video2rig-colab.zip 源码包")
    source_zip = upload_one()
    with zipfile.ZipFile(source_zip) as bundle:
        for item in bundle.infolist():
            if not (session / item.filename).resolve().is_relative_to(session) or '\\' in item.filename:
                raise ValueError("非法源码 ZIP 路径")
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("源码 ZIP 不接受符号链接")
        bundle.extractall(session)
    repo = session / 'video2rig'
else:
    repo = session / 'video2rig'
    run(['git', 'clone', '--depth', '1', '--branch', repo_ref, repo_url, repo])
if not (repo / 'train_colab.py').is_file():
    raise RuntimeError("此版本尚未包含 Colab 配置；请使用交付的源码 ZIP，或选择已合入配置的分支")
run([sys.executable, '-m', 'pip', 'install', '-r', repo / 'requirements-colab.txt'])


In [ ]:
#@title 上传训练数据并一键训练
archive = Path(training_zip).expanduser().resolve() if training_zip.strip() else None
if archive is None:
    print("请上传本地 pack 生成的 training.zip")
    archive = upload_one()
run([sys.executable, repo / 'train_colab.py', 'unpack', archive, session / 'data'])
project = session / 'data' / 'project'
result_dir = Path('/content/drive/MyDrive/video2rig-results') if use_drive else session
result_dir.mkdir(parents=True, exist_ok=True)
result_zip = result_dir / (session.name + '-reader.zip')
run([sys.executable, '-u', repo / 'train_colab.py', 'train', project,
     '--steps', steps, '--minutes', minutes, '--batch', batch, '--seed', seed,
     '--result', result_zip])


In [ ]:
#@title 下载结果
import json
print((project / 'reader' / 'reader-train.json').read_text())
print("结果 ZIP:", result_zip)
files.download(str(result_zip))


## 回到原 Python / Blender 流程
把结果 ZIP 解压到原来的 creature 项目根目录，得到 `my_troll/reader/reader.pt` 和 `reader-train.json`，然后继续：
```bash
python -m video2rig run my_troll stagger
```
显存不足：将 batch 降到 2（或 1）。T4 使用 FP32 以保证兼容性，速度和内存用量与 BF16 卡不同。
Colab 断线不会自动保存中间进度；权重在训练结束时保存。启用 Drive 可保存最终 ZIP，但不等同于断点续训。
`--init` 是权重热启动，不恢复优化器或训练步数。首次训练需要访问 Hugging Face 下载 DINOv2。